In [0]:
from datetime import datetime, timezone
from pathlib import PurePosixPath
from uuid import uuid4

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    TimestampType,
    BooleanType
)

BACKUP_CATALOG = "backup_admin"
BACKUP_SCHEMA = "volume_protection"
BACKUP_VOLUME = "backups"

POLICY_TABLE = (
    BACKUP_CATALOG
    + "."
    + BACKUP_SCHEMA
    + ".volume_backup_policy"
)

MANIFEST_TABLE = (
    BACKUP_CATALOG
    + "."
    + BACKUP_SCHEMA
    + ".volume_backup_manifest"
)

RUN_TABLE = (
    BACKUP_CATALOG
    + "."
    + BACKUP_SCHEMA
    + ".volume_backup_run"
)

RESTORE_AUDIT_TABLE = (
    BACKUP_CATALOG
    + "."
    + BACKUP_SCHEMA
    + ".volume_restore_audit"
)

BACKUP_VOLUME_PATH = (
    "/Volumes/"
    + BACKUP_CATALOG
    + "/"
    + BACKUP_SCHEMA
    + "/"
    + BACKUP_VOLUME
)

RESTORE_STAGING_ROOT = (
    BACKUP_VOLUME_PATH
    + "/restore_staging"
)

print("Volume recovery framework")
print("=" * 70)
print("Policy table       :", POLICY_TABLE)
print("Manifest table     :", MANIFEST_TABLE)
print("Restore audit table:", RESTORE_AUDIT_TABLE)
print("Restore staging    :", RESTORE_STAGING_ROOT)

In [0]:
spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {RESTORE_AUDIT_TABLE}
    (
        restore_id STRING NOT NULL,
        restore_timestamp TIMESTAMP NOT NULL,
        policy_id STRING,
        snapshot_id STRING,
        source_catalog STRING,
        source_schema STRING,
        source_volume STRING,
        source_relative_path STRING,
        backup_full_path STRING,
        restore_target_path STRING,
        restore_mode STRING,
        target_mode STRING,
        overwrite_enabled BOOLEAN,
        dry_run BOOLEAN,
        file_size_bytes BIGINT,
        restore_status STRING,
        error_message STRING,
        restored_by STRING,
        created_at TIMESTAMP
    )
    USING DELTA
    COMMENT 'Audit history for point-in-time Unity Catalog volume file restoration'
    """
)

print(
    "Restore audit table is ready:",
    RESTORE_AUDIT_TABLE
)

In [0]:
def utc_now():
    return datetime.now(timezone.utc)
def canonical_volume_path(path):

    if path is None:
        return None

    normalized = normalize_volume_path(
        path
    )

    volume_prefix = "/Volumes/"

    if normalized.startswith(
        volume_prefix
    ):

        remainder = normalized[
            len(volume_prefix):
        ]

        while "//" in remainder:

            remainder = remainder.replace(
                "//",
                "/"
            )

        return (
            volume_prefix
            + remainder
        )

    while "//" in normalized:

        normalized = normalized.replace(
            "//",
            "/"
        )

    return normalized

def sql_string(value):

    if value is None:
        return "NULL"

    escaped = str(value).replace(
        "'",
        "''"
    )

    return "'" + escaped + "'"


def normalize_volume_path(path):

    if path is None:
        return None

    if path.startswith("dbfs:/Volumes/"):

        return path.replace(
            "dbfs:/Volumes/",
            "/Volumes/",
            1
        )

    return path
def path_exists(path):

    if not path:
        return False

    clean_path = canonical_volume_path(
        path
    )

    try:

        dbutils.fs.ls(
            clean_path
        )

        return True

    except Exception:

        return False


def clean_volume_path(path):

    if path is None:
        return None

    normalized = normalize_volume_path(
        path
    )

    prefix = "/Volumes/"

    if normalized.startswith(prefix):

        remainder = normalized[
            len(prefix):
        ]

        while "//" in remainder:
            remainder = remainder.replace(
                "//",
                "/"
            )

        return prefix + remainder

    while "//" in normalized:
        normalized = normalized.replace(
            "//",
            "/"
        )

    return normalized


def safe_path_component(value):

    allowed = set(
        "abcdefghijklmnopqrstuvwxyz"
        "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
        "0123456789"
        "-_."
    )

    result = ""

    for character in str(value):

        if character in allowed:
            result += character
        else:
            result += "_"

    return result


def path_exists(path):

    if not path:
        return False

    clean_path = canonical_volume_path(
        path
    )

    try:

        dbutils.fs.ls(
            clean_path
        )

        return True

    except Exception:

        return False



def parent_directory(path):

    normalized = clean_volume_path(
        path
    )

    return str(
        PurePosixPath(
            normalized
        ).parent
    )


def ensure_parent_directory(path):

    directory = parent_directory(
        path
    )

    dbutils.fs.mkdirs(
        directory
    )


def current_user_name():

    try:

        return spark.sql(
            """
            SELECT current_user()
                AS user_name
            """
        ).first()["user_name"]

    except Exception:

        return "UNKNOWN"


print(
    "Restore helper functions initialized successfully."
)

In [0]:
try:
    dbutils.widgets.removeAll()
except Exception:
    pass


dbutils.widgets.text(
    "policy_id",
    "POLICY_APP_ARTIFACTS",
    "Backup policy ID"
)

dbutils.widgets.text(
    "snapshot_id",
    "",
    "Snapshot ID"
)

dbutils.widgets.dropdown(
    "restore_mode",
    "SNAPSHOT",
    [
        "SNAPSHOT",
        "SINGLE_FILE"
    ],
    "Restore mode"
)

dbutils.widgets.text(
    "relative_file_path",
    "",
    "Relative file path"
)

dbutils.widgets.dropdown(
    "target_mode",
    "STAGING",
    [
        "STAGING",
        "SOURCE"
    ],
    "Restore target"
)

dbutils.widgets.dropdown(
    "dry_run",
    "true",
    [
        "true",
        "false"
    ],
    "Preview only"
)

dbutils.widgets.dropdown(
    "overwrite_existing",
    "false",
    [
        "true",
        "false"
    ],
    "Overwrite existing files"
)


POLICY_ID = (
    dbutils.widgets
    .get("policy_id")
    .strip()
)

SNAPSHOT_ID = (
    dbutils.widgets
    .get("snapshot_id")
    .strip()
)

RESTORE_MODE = (
    dbutils.widgets
    .get("restore_mode")
    .strip()
    .upper()
)

RELATIVE_FILE_PATH = (
    dbutils.widgets
    .get("relative_file_path")
    .strip()
)

TARGET_MODE = (
    dbutils.widgets
    .get("target_mode")
    .strip()
    .upper()
)

DRY_RUN = (
    dbutils.widgets
    .get("dry_run")
    .strip()
    .lower()
    == "true"
)

OVERWRITE_EXISTING = (
    dbutils.widgets
    .get("overwrite_existing")
    .strip()
    .lower()
    == "true"
)

print("Policy ID          :", POLICY_ID)
print("Snapshot ID        :", SNAPSHOT_ID or "Not selected")
print("Restore mode       :", RESTORE_MODE)
print("Relative file path :", RELATIVE_FILE_PATH or "Not supplied")
print("Target mode        :", TARGET_MODE)
print("Dry run            :", DRY_RUN)
print("Overwrite existing :", OVERWRITE_EXISTING)

In [0]:
policies_df = spark.sql(
    f"""
    SELECT
        policy_id,
        source_catalog,
        source_schema,
        source_volume,
        backup_enabled,
        backup_priority,
        retention_days,
        backup_frequency,
        description
    FROM {POLICY_TABLE}
    WHERE backup_enabled = true
    ORDER BY
        backup_priority,
        source_catalog,
        source_schema,
        source_volume
    """
)

display(
    policies_df
)

In [0]:
if not POLICY_ID:

    print(
        "Select a policy first."
    )

else:

    snapshots_df = spark.sql(
        f"""
        SELECT
            snapshot_id,

            MAX(snapshot_timestamp)
                AS snapshot_timestamp,

            COUNT(*)
                AS manifest_record_count,

            SUM(
                CASE
                    WHEN change_type = 'NEW'
                    THEN 1 ELSE 0
                END
            ) AS new_files,

            SUM(
                CASE
                    WHEN change_type = 'MODIFIED'
                    THEN 1 ELSE 0
                END
            ) AS modified_files,

            SUM(
                CASE
                    WHEN change_type = 'UNCHANGED'
                    THEN 1 ELSE 0
                END
            ) AS unchanged_files,

            SUM(
                CASE
                    WHEN change_type = 'DELETED'
                    THEN 1 ELSE 0
                END
            ) AS deleted_files,

            SUM(
                CASE
                    WHEN backup_status = 'FAILED'
                    THEN 1 ELSE 0
                END
            ) AS failed_files

        FROM {MANIFEST_TABLE}

        WHERE policy_id =
              {sql_string(POLICY_ID)}

        GROUP BY snapshot_id

        ORDER BY snapshot_timestamp DESC
        """
    )

    display(
        snapshots_df
    )

In [0]:
if not POLICY_ID:

    raise ValueError(
        "policy_id must be supplied."
    )


if not SNAPSHOT_ID:

    raise ValueError(
        "snapshot_id must be supplied."
    )


if RESTORE_MODE not in {
    "SNAPSHOT",
    "SINGLE_FILE"
}:

    raise ValueError(
        "restore_mode must be SNAPSHOT "
        "or SINGLE_FILE."
    )


if TARGET_MODE not in {
    "STAGING",
    "SOURCE"
}:

    raise ValueError(
        "target_mode must be STAGING "
        "or SOURCE."
    )


if (
    RESTORE_MODE == "SINGLE_FILE"
    and not RELATIVE_FILE_PATH
):

    raise ValueError(
        "relative_file_path is required "
        "for SINGLE_FILE mode."
    )


if (
    TARGET_MODE == "SOURCE"
    and DRY_RUN is False
    and OVERWRITE_EXISTING is False
):

    print(
        "Source restore is enabled, but existing "
        "files will not be overwritten."
    )


policy_row = spark.sql(
    f"""
    SELECT
        policy_id,
        source_catalog,
        source_schema,
        source_volume,
        retention_days,
        description
    FROM {POLICY_TABLE}
    WHERE policy_id =
          {sql_string(POLICY_ID)}
      AND backup_enabled = true
    LIMIT 1
    """
).first()


if policy_row is None:

    raise ValueError(
        "No enabled backup policy found for: "
        + POLICY_ID
    )


SOURCE_CATALOG = (
    policy_row["source_catalog"]
)

SOURCE_SCHEMA = (
    policy_row["source_schema"]
)

SOURCE_VOLUME = (
    policy_row["source_volume"]
)


SOURCE_VOLUME_PATH = (
    "/Volumes/"
    + SOURCE_CATALOG
    + "/"
    + SOURCE_SCHEMA
    + "/"
    + SOURCE_VOLUME
)


print("Validated restore policy")
print("-" * 70)
print("Policy       :", POLICY_ID)
print("Source volume:", SOURCE_VOLUME_PATH)
print("Snapshot     :", SNAPSHOT_ID)
print("Restore mode :", RESTORE_MODE)
print("Target mode  :", TARGET_MODE)

In [0]:
snapshot_header = spark.sql(
    f"""
    SELECT
        snapshot_id,
        MAX(snapshot_timestamp)
            AS snapshot_timestamp,
        COUNT(*)
            AS record_count
    FROM {MANIFEST_TABLE}
    WHERE policy_id =
          {sql_string(POLICY_ID)}
      AND snapshot_id =
          {sql_string(SNAPSHOT_ID)}
    GROUP BY snapshot_id
    """
).first()


if snapshot_header is None:

    raise ValueError(
        "The selected snapshot was not found. "
        + "Policy="
        + POLICY_ID
        + ", Snapshot="
        + SNAPSHOT_ID
    )


SNAPSHOT_TIMESTAMP = (
    snapshot_header[
        "snapshot_timestamp"
    ]
)


print(
    "Snapshot found:"
)

print(
    SNAPSHOT_ID
)

print(
    "Snapshot timestamp:",
    SNAPSHOT_TIMESTAMP
)

print(
    "Manifest records:",
    snapshot_header["record_count"]
)

In [0]:
restore_filter = f"""
WHERE policy_id =
      {sql_string(POLICY_ID)}

  AND snapshot_id =
      {sql_string(SNAPSHOT_ID)}

  AND backup_status = 'SUCCESS'

  AND change_type <> 'DELETED'
"""


if RESTORE_MODE == "SINGLE_FILE":

    restore_filter += (
        "\nAND source_relative_path = "
        + sql_string(
            RELATIVE_FILE_PATH
        )
    )


restore_rows = spark.sql(
    f"""
    SELECT
        snapshot_id,
        policy_id,
        snapshot_timestamp,
        source_catalog,
        source_schema,
        source_volume,
        source_relative_path,
        source_full_path,
        backup_full_path,
        file_name,
        file_size_bytes,
        source_modification_time,
        file_checksum,
        change_type,
        backup_status

    FROM {MANIFEST_TABLE}

    {restore_filter}

    ORDER BY source_relative_path
    """
).collect()


if not restore_rows:

    raise ValueError(
        "No recoverable files were found for "
        "the selected restore criteria."
    )


RESTORE_ID = str(
    uuid4()
)


if TARGET_MODE == "STAGING":

    RESTORE_TARGET_ROOT = (
        RESTORE_STAGING_ROOT
        + "/"
        + safe_path_component(
            POLICY_ID
        )
        + "/"
        + safe_path_component(
            SNAPSHOT_ID
        )
        + "/"
        + RESTORE_ID
    )

else:

    RESTORE_TARGET_ROOT = (
        SOURCE_VOLUME_PATH
    )


RESTORE_TARGET_ROOT = (
    canonical_volume_path(
        RESTORE_TARGET_ROOT
    )
)


restore_plan = []


for row in restore_rows:

    relative_path = row[
        "source_relative_path"
    ]

    backup_path = canonical_volume_path(
        row[
            "backup_full_path"
        ]
    )

    target_path = (
        RESTORE_TARGET_ROOT.rstrip("/")
        + "/"
        + relative_path.lstrip("/")
    )

    target_path = canonical_volume_path(
        target_path
    )


    backup_object_exists = path_exists(
        backup_path
    )

    target_already_exists = path_exists(
        target_path
    )


    planned_action = "RESTORE"

    validation_error = None


    if not backup_object_exists:

        planned_action = (
            "BACKUP_OBJECT_MISSING"
        )

        validation_error = (
            "Backup object does not exist: "
            + backup_path
        )

    elif (
        target_already_exists
        and not OVERWRITE_EXISTING
    ):

        planned_action = (
            "SKIP_TARGET_EXISTS"
        )


    restore_plan.append(
        {
            "restore_id":
                RESTORE_ID,

            "snapshot_id":
                SNAPSHOT_ID,

            "policy_id":
                POLICY_ID,

            "source_relative_path":
                relative_path,

            "backup_full_path":
                backup_path,

            "restore_target_path":
                target_path,

            "file_size_bytes":
                int(
                    row[
                        "file_size_bytes"
                    ] or 0
                ),

            "change_type":
                row[
                    "change_type"
                ],

            "target_exists":
                target_already_exists,

            "backup_object_exists":
                backup_object_exists,

            "planned_action":
                planned_action,

            "validation_error":
                validation_error
        }
    )


restore_plan_df = spark.createDataFrame(
    restore_plan
)


display(
    restore_plan_df
)

In [0]:
total_files = len(
    restore_plan
)

restorable_files = sum(
    1
    for item in restore_plan
    if item["planned_action"]
    == "RESTORE"
)

existing_target_files = sum(
    1
    for item in restore_plan
    if item["planned_action"]
    == "SKIP_TARGET_EXISTS"
)

missing_backup_objects = sum(
    1
    for item in restore_plan
    if item["planned_action"]
    == "BACKUP_OBJECT_MISSING"
)

total_restore_bytes = sum(
    item["file_size_bytes"]
    for item in restore_plan
    if item["planned_action"]
    == "RESTORE"
)


print("=" * 70)
print("RESTORE PLAN")
print("=" * 70)

print(
    "Restore ID           :",
    RESTORE_ID
)

print(
    "Policy ID            :",
    POLICY_ID
)

print(
    "Snapshot ID          :",
    SNAPSHOT_ID
)

print(
    "Snapshot timestamp   :",
    SNAPSHOT_TIMESTAMP
)

print(
    "Restore mode         :",
    RESTORE_MODE
)

print(
    "Target mode          :",
    TARGET_MODE
)

print(
    "Target root          :",
    RESTORE_TARGET_ROOT
)

print(
    "Dry run              :",
    DRY_RUN
)

print(
    "Overwrite existing   :",
    OVERWRITE_EXISTING
)

print("-" * 70)

print(
    "Files in plan        :",
    total_files
)

print(
    "Files restorable     :",
    restorable_files
)

print(
    "Existing targets     :",
    existing_target_files
)

print(
    "Missing backups      :",
    missing_backup_objects
)

print(
    "Bytes to restore     :",
    f"{total_restore_bytes:,}"
)

In [0]:
if missing_backup_objects > 0:

    raise RuntimeError(
        str(missing_backup_objects)
        + " backup objects are missing. "
        + "Restore execution has been blocked."
    )


if (
    TARGET_MODE == "SOURCE"
    and not DRY_RUN
):

    print(
        "WARNING: This execution can write into "
        "the original source volume."
    )


if (
    TARGET_MODE == "SOURCE"
    and not DRY_RUN
    and OVERWRITE_EXISTING
):

    print(
        "CRITICAL WARNING: Existing source files "
        "can be overwritten."
    )


if DRY_RUN:

    print()
    print(
        "DRY RUN is enabled."
    )

    print(
        "No files will be copied."
    )

In [0]:
restore_audit_schema = StructType(
    [
        StructField(
            "restore_id",
            StringType(),
            False
        ),
        StructField(
            "restore_timestamp",
            TimestampType(),
            False
        ),
        StructField(
            "policy_id",
            StringType(),
            True
        ),
        StructField(
            "snapshot_id",
            StringType(),
            True
        ),
        StructField(
            "source_catalog",
            StringType(),
            True
        ),
        StructField(
            "source_schema",
            StringType(),
            True
        ),
        StructField(
            "source_volume",
            StringType(),
            True
        ),
        StructField(
            "source_relative_path",
            StringType(),
            True
        ),
        StructField(
            "backup_full_path",
            StringType(),
            True
        ),
        StructField(
            "restore_target_path",
            StringType(),
            True
        ),
        StructField(
            "restore_mode",
            StringType(),
            True
        ),
        StructField(
            "target_mode",
            StringType(),
            True
        ),
        StructField(
            "overwrite_enabled",
            StringType(),
            True
        ),
        StructField(
            "dry_run",
            StringType(),
            True
        ),
        StructField(
            "file_size_bytes",
            LongType(),
            True
        ),
        StructField(
            "restore_status",
            StringType(),
            True
        ),
        StructField(
            "error_message",
            StringType(),
            True
        ),
        StructField(
            "restored_by",
            StringType(),
            True
        ),
        StructField(
            "created_at",
            TimestampType(),
            True
        )
    ]
)

In [0]:
display(
    spark.sql(
        f"""
        SELECT
            restore_id,
            restore_timestamp,
            policy_id,
            snapshot_id,
            source_relative_path,
            backup_full_path,
            restore_target_path,
            restore_mode,
            target_mode,
            overwrite_enabled,
            dry_run,
            file_size_bytes,
            restore_status,
            error_message,
            restored_by
        FROM {RESTORE_AUDIT_TABLE}
        WHERE restore_id =
              {sql_string(RESTORE_ID)}
        ORDER BY source_relative_path
        """
    )
)

In [0]:
if (
    TARGET_MODE == "STAGING"
    and not DRY_RUN
):

    print(
        "Files restored to staging:"
    )

    display(
        dbutils.fs.ls(
            RESTORE_TARGET_ROOT
        )
    )

else:

    print(
        "Staging validation is not applicable "
        "for this execution."
    )

In [0]:
print("=" * 70)
print("VOLUME RESTORE SUMMARY")
print("=" * 70)

print(
    "Restore ID         :",
    RESTORE_ID
)

print(
    "Policy ID          :",
    POLICY_ID
)

print(
    "Snapshot ID        :",
    SNAPSHOT_ID
)

print(
    "Snapshot timestamp :",
    SNAPSHOT_TIMESTAMP
)

print(
    "Restore mode       :",
    RESTORE_MODE
)

print(
    "Target mode        :",
    TARGET_MODE
)

print(
    "Dry run            :",
    DRY_RUN
)

print(
    "Overwrite enabled  :",
    OVERWRITE_EXISTING
)

print(
    "Restored files     :",
    restored_file_count
)

print(
    "Skipped files      :",
    skipped_file_count
)

print(
    "Failed files       :",
    failed_file_count
)

print(
    "Restored bytes     :",
    f"{restored_bytes:,}"
)

print(
    "Target root        :",
    RESTORE_TARGET_ROOT
)


if DRY_RUN:

    print()
    print(
        "This was a dry run. No files were copied."
    )


if failed_file_count > 0:

    raise RuntimeError(
        str(failed_file_count)
        + " file restore operations failed. "
        + "Restore ID: "
        + RESTORE_ID
    )

In [0]:
backup_root = (
    "/Volumes/backup_admin/"
    "volume_protection/backups/objects/"
    "royalinsurance/edw/app_artifacts"
)

entries = dbutils.fs.ls(
    backup_root
)

for entry in entries:

    if entry.isDir():

        directory_path = clean_volume_path(
            entry.path
        ).rstrip("/")

        print("=" * 80)
        print("DIRECTORY:")
        print(directory_path)

        try:

            children = dbutils.fs.ls(
                directory_path
            )

            if not children:
                print("EMPTY DIRECTORY")

            for child in children:

                print(
                    "FILE:",
                    clean_volume_path(
                        child.path
                    )
                )

                print(
                    "SIZE:",
                    child.size
                )

        except Exception as error:

            print(
                "ERROR:",
                str(error)
            )